# Phase 1: Load and explore the data

Goal: understand the Asclepius synthetic clinical notes before modeling anything.
Each step below follows Phase 1 of the build guide.

## 1. Load the data and save a local copy
About 400 MB. The first download takes a few minutes; later runs use the Hugging Face cache.

In [ ]:
from pathlib import Path

import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 80)

DATA = Path("data")
(DATA / "raw").mkdir(parents=True, exist_ok=True)

ds = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes", split="train")
df = ds.to_pandas()
df.to_parquet(DATA / "raw" / "asclepius.parquet")
print(df.shape)  # expect about (158114, 5)
df.head()

## 2. Count questions per task

In [ ]:
df["task"].value_counts()

## 3. Measure lengths
This shows whether a note fits in a small model's context.

In [ ]:
df["note_words"] = df["note"].str.split().str.len()
df["answer_words"] = df["answer"].str.split().str.len()

display(df.groupby("task")[["note_words", "answer_words"]].median())
display(df["note_words"].describe())

## 4. Check how questions are spread across patients and notes

In [ ]:
display(df.groupby("patient_id").size().describe())
print("duplicated notes:", df["note"].duplicated().sum())  # identical notes under different rows?

## 5. Read real examples
This matters more than any statistic. Read at least 15, including some from every task.
While reading, note how section headers are written (for example, "Hospital Course:"). You'll need that pattern in Phase 4.

In [ ]:
for _, r in df.sample(5, random_state=1).iterrows():
    print("TASK:", r.task)
    print("NOTE:", r.note[:1500])
    print("Q:", r.question)
    print("A:", r.answer)
    print("-" * 80)

One example from every task, so each task gets read at least once:

In [ ]:
for task, g in df.groupby("task"):
    r = g.sample(1, random_state=2).iloc[0]
    print("TASK:", task)
    print("NOTE:", r.note[:1000])
    print("Q:", r.question)
    print("A:", r.answer)
    print("-" * 80)

## 6. Save a 5,000-row development sample, balanced across tasks

In [ ]:
n_per_task = 5000 // df["task"].nunique()
# groupby().sample keeps the "task" column; groupby().apply drops it in pandas 3
sample = df.groupby("task").sample(n=n_per_task, random_state=42)
sample.to_parquet(DATA / "sample.parquet")
print(sample.shape)
sample["task"].value_counts()

## 7. Write down what you found

Put these in `notes.md`: the task counts, typical note length, anything odd, and a few wrong reference answers if you spot them. This becomes the Data section of the README.

**Checkpoint:** you can explain in two sentences what the eight tasks are and how long a typical note is.